# Reflection Removal — Training Pipeline

This notebook trains the full reflection removal model in three stages:

| Stage | Description | Saves |
|-------|-------------|-------|
| 0 | Baseline ResNet-UNet (no refraction module) | `baseline.pth` |
| 1 | Full model — synthetic pre-training | `improved_model_stage1.pth` |
| 2 | Real fine-tuning with perceptual + SSIM + edge loss | `final_model_v3.pth` |

**Dataset layout expected:**
```
/content/drive/MyDrive/Reflection_Removal/
  datasets/
    SIR2/training_pairs/        ← A/ (blended), B/ (ground-truth)
    Perceptual/synthetic/       ← blended/, transmission_layer_clean/
  models/                       ← checkpoints will be saved here
```


## 0. Install Dependencies

In [ ]:
!pip install -q torchvision scikit-image pytorch-msssim


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Imports & Device

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 3. Configure Paths

In [ ]:
# ── Edit these paths if your Drive layout differs ──
SYN_PATH   = "/content/drive/MyDrive/Reflection_Removal/datasets/Perceptual/synthetic"
REAL_PATH  = "/content/drive/MyDrive/Reflection_Removal/datasets/SIR2/training_pairs"
MODEL_DIR  = "/content/drive/MyDrive/Reflection_Removal/models"

os.makedirs(MODEL_DIR, exist_ok=True)


## 4. Datasets

In [ ]:
transform = T.Compose([T.Resize((256, 256)), T.ToTensor()])


class SyntheticDataset(Dataset):
    """Perceptual synthetic dataset — blended / transmission_layer_clean."""

    def __init__(self, root_dir):
        self.input_dir = os.path.join(root_dir, "blended")
        self.gt_dir    = os.path.join(root_dir, "transmission_layer_clean")
        self.files     = sorted(os.listdir(self.input_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f   = self.files[idx]
        inp = Image.open(os.path.join(self.input_dir, f)).convert("RGB")
        gt  = Image.open(os.path.join(self.gt_dir,    f)).convert("RGB")
        return transform(inp), transform(gt)


class RealDataset(Dataset):
    """SIR2 real dataset — A/ (blended), B/ (ground-truth)."""

    def __init__(self, root_dir):
        self.input_dir = os.path.join(root_dir, "A")
        self.gt_dir    = os.path.join(root_dir, "B")
        self.files     = sorted(os.listdir(self.input_dir))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f   = self.files[idx]
        inp = Image.open(os.path.join(self.input_dir, f)).convert("RGB")
        gt  = Image.open(os.path.join(self.gt_dir,    f)).convert("RGB")
        return transform(inp), transform(gt)


syn_loader  = DataLoader(SyntheticDataset(SYN_PATH),  batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
real_loader = DataLoader(RealDataset(REAL_PATH),       batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)

print(f"Synthetic samples : {len(syn_loader.dataset)}")
print(f"Real samples      : {len(real_loader.dataset)}")


## 5. Model Architecture

In [ ]:
# ── Refraction (optical flow alignment) module ──────────────────────────────

class RefractionModule(nn.Module):
    """
    Predicts a 2-D displacement field and warps the input image accordingly.
    This corrects for glass / mirror refraction before the denoising UNet.
    """

    def __init__(self):
        super().__init__()
        self.flow_predictor = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32,  2, 3, padding=1),          # (dx, dy) per pixel
        )

    def forward(self, x):
        flow = self.flow_predictor(x)                  # (B, 2, H, W)
        B, C, H, W = x.shape

        grid_y, grid_x = torch.meshgrid(
            torch.linspace(-1, 1, H, device=x.device),
            torch.linspace(-1, 1, W, device=x.device),
            indexing='ij'
        )
        grid = torch.stack((grid_x, grid_y), dim=2)   # (H, W, 2)
        grid = grid.unsqueeze(0).expand(B, -1, -1, -1) # (B, H, W, 2)

        flow = flow.permute(0, 2, 3, 1)               # (B, H, W, 2)
        flow = flow / torch.tensor([W, H], dtype=torch.float32, device=x.device)

        warped = F.grid_sample(x, grid + flow, align_corners=True)
        return warped, flow


# ── ResNet-50 encoder / decoder UNet ────────────────────────────────────────

class ResNetUNet(nn.Module):
    """
    ResNet-50 encoder with a lightweight transposed-convolution decoder.
    Output is clamped to [0, 1] via sigmoid and interpolated back to
    the input resolution.
    """

    def __init__(self):
        super().__init__()
        base   = models.resnet50(weights="IMAGENET1K_V1")
        layers = list(base.children())

        # Encoder
        self.layer0 = nn.Sequential(*layers[:3])   # stride 2  → 64ch
        self.layer1 = nn.Sequential(*layers[3:5])  # stride 4  → 256ch
        self.layer2 = layers[5]                    # stride 8  → 512ch
        self.layer3 = layers[6]                    # stride 16 → 1024ch
        self.layer4 = layers[7]                    # stride 32 → 2048ch

        # Decoder
        self.up4 = nn.ConvTranspose2d(2048, 1024, 2, 2)
        self.up3 = nn.ConvTranspose2d(1024,  512, 2, 2)
        self.up2 = nn.ConvTranspose2d( 512,  256, 2, 2)
        self.up1 = nn.ConvTranspose2d( 256,   64, 2, 2)
        self.final = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        out = self.final(self.up1(self.up2(self.up3(self.up4(x4)))))

        # Restore spatial size and squash to [0,1]
        out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return torch.sigmoid(out)


# ── Full model ───────────────────────────────────────────────────────────────

class ReflectionRemovalModel(nn.Module):
    """Full pipeline: RefractionModule → ResNetUNet."""

    def __init__(self):
        super().__init__()
        self.refraction = RefractionModule()
        self.unet       = ResNetUNet()

    def forward(self, x):
        aligned, _ = self.refraction(x)
        out        = self.unet(aligned)
        return out, aligned


# Quick shape sanity check
_x = torch.randn(1, 3, 256, 256)
_m = ReflectionRemovalModel()
_out, _aln = _m(_x)
print("Output shape  :", _out.shape)   # expect (1, 3, 256, 256)
print("Aligned shape :", _aln.shape)   # expect (1, 3, 256, 256)
del _x, _m, _out, _aln


## 6. Loss Functions

In [ ]:
from pytorch_msssim import ssim as ms_ssim

# Freeze VGG-16 feature extractor for perceptual loss
vgg = models.vgg16(weights="IMAGENET1K_V1").features[:16].to(device).eval()
for p in vgg.parameters():
    p.requires_grad = False


def perceptual_loss(pred, target):
    """L1 loss in VGG-16 feature space."""
    return F.l1_loss(vgg(pred), vgg(target))


def edge_loss(pred, target):
    """Sobel-gradient L1 loss — encourages sharp edges."""
    sobel_x = torch.tensor([[1, 0,-1],[2, 0,-2],[1, 0,-1]], dtype=torch.float32,
                             device=pred.device).view(1,1,3,3)
    sobel_y = torch.tensor([[1, 2, 1],[0, 0, 0],[-1,-2,-1]], dtype=torch.float32,
                             device=pred.device).view(1,1,3,3)

    def grad(img):
        channels = []
        for c in range(3):
            ch = img[:, c:c+1]
            gx = F.conv2d(ch, sobel_x, padding=1)
            gy = F.conv2d(ch, sobel_y, padding=1)
            channels.append(torch.sqrt(gx**2 + gy**2 + 1e-6))
        return torch.cat(channels, dim=1)

    return F.l1_loss(grad(pred), grad(target))


def combined_loss(pred, target, aligned):
    """
    Weighted combination of L1, alignment, perceptual, SSIM, and edge losses.
    Sizes are matched via interpolation to avoid shape mismatches.
    """
    if pred.shape != target.shape:
        pred    = F.interpolate(pred,    size=target.shape[2:], mode='bilinear', align_corners=False)
    if aligned.shape != target.shape:
        aligned = F.interpolate(aligned, size=target.shape[2:], mode='bilinear', align_corners=False)

    l1         = F.l1_loss(pred, target)
    align      = F.l1_loss(aligned, target)
    perceptual = perceptual_loss(pred, target)
    ssim_loss  = 1 - ms_ssim(pred, target, data_range=1.0)
    edge       = edge_loss(pred, target)

    return l1 + 0.01 * align + 0.1 * perceptual + 0.2 * ssim_loss + 0.1 * edge


## 7. Checkpoint Utilities

In [ ]:
def safe_load(model, path):
    """Load a checkpoint, skipping any layers whose shapes don't match."""
    checkpoint  = torch.load(path, map_location=device)
    model_state = model.state_dict()

    compatible = {
        k: v for k, v in checkpoint.items()
        if k in model_state and v.shape == model_state[k].shape
    }

    model_state.update(compatible)
    model.load_state_dict(model_state)
    print(f"  Loaded {len(compatible)}/{len(model_state)} layers from '{path}'")
    return model


## 8. Stage 0 — Baseline Training (ResNet-UNet only)

Train a plain ResNet-UNet **without** the refraction module on real data.
This checkpoint is later used as the encoder initialisation for Stage 1.


In [ ]:
# ── Standalone UNet model for baseline ──────────────────────────────────────
baseline_model = ResNetUNet().to(device)
baseline_optim = torch.optim.AdamW(baseline_model.parameters(), lr=1e-4)

BASELINE_EPOCHS = 5
BASELINE_CKPT   = os.path.join(MODEL_DIR, "baseline.pth")

print("=== Stage 0: Baseline Training ===")

for epoch in range(BASELINE_EPOCHS):
    baseline_model.train()
    total_loss = 0.0

    for inp, gt in real_loader:
        inp, gt = inp.to(device), gt.to(device)

        pred = baseline_model(inp)
        loss = F.l1_loss(pred, gt)

        baseline_optim.zero_grad()
        loss.backward()
        baseline_optim.step()

        total_loss += loss.item()

    avg = total_loss / len(real_loader)
    print(f"  Epoch {epoch+1}/{BASELINE_EPOCHS} — Loss: {avg:.4f}")
    torch.save(baseline_model.state_dict(), BASELINE_CKPT)

print(f"Baseline saved → {BASELINE_CKPT}")


## 9. Stage 1 — Synthetic Pre-Training (Full Model)

Initialise the UNet encoder from the baseline checkpoint, then train the
full pipeline (RefractionModule + UNet) on the larger synthetic dataset.


In [ ]:
model = ReflectionRemovalModel().to(device)

# Transfer baseline weights into the UNet sub-module
model.unet = safe_load(model.unet, BASELINE_CKPT)

# Differential learning rates: slower for the pretrained encoder
optimizer = torch.optim.AdamW([
    {"params": model.unet.parameters(),       "lr": 5e-5},
    {"params": model.refraction.parameters(), "lr": 1e-4},
])

STAGE1_EPOCHS = 5
STAGE1_CKPT   = os.path.join(MODEL_DIR, "improved_model_stage1.pth")

print("=== Stage 1: Synthetic Pre-Training ===")

for epoch in range(STAGE1_EPOCHS):
    model.train()
    total_loss = 0.0

    for i, (inp, gt) in enumerate(syn_loader):
        inp, gt = inp.to(device), gt.to(device)

        pred, aligned = model(inp)
        loss = combined_loss(pred, gt, aligned)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 100 == 0:
            print(f"  Epoch {epoch+1} | Batch {i}/{len(syn_loader)} | Loss: {loss.item():.4f}")

    avg = total_loss / len(syn_loader)
    print(f"  Epoch {epoch+1}/{STAGE1_EPOCHS} complete — Avg Loss: {avg:.4f}")
    torch.save(model.state_dict(), STAGE1_CKPT)

print(f"Stage 1 checkpoint saved → {STAGE1_CKPT}")


## 10. Stage 2 — Real Data Fine-Tuning

Fine-tune on real (SIR2) data using the full combined loss, including
perceptual, SSIM, and edge terms for sharper and more realistic output.


In [ ]:
# Load Stage 1 weights
model = safe_load(model, STAGE1_CKPT)

# Lower LR for fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

STAGE2_EPOCHS = 3
FINAL_CKPT    = os.path.join(MODEL_DIR, "final_model_v3.pth")

print("=== Stage 2: Real Data Fine-Tuning ===")

for epoch in range(STAGE2_EPOCHS):
    model.train()
    total_loss = 0.0

    for i, (inp, gt) in enumerate(real_loader):
        inp, gt = inp.to(device), gt.to(device)

        pred, aligned = model(inp)
        loss = combined_loss(pred, gt, aligned)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if i % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {i}/{len(real_loader)} | Loss: {loss.item():.4f}")

    avg = total_loss / len(real_loader)
    print(f"  Epoch {epoch+1}/{STAGE2_EPOCHS} complete — Avg Loss: {avg:.4f}")
    torch.save(model.state_dict(), FINAL_CKPT)

print(f"Final model saved → {FINAL_CKPT}")


## Training Complete

All checkpoints saved under `MODEL_DIR`:
- `baseline.pth` — Stage 0 (ResNetUNet only)
- `improved_model_stage1.pth` — Stage 1 (synthetic pre-training)
- `final_model_v3.pth` — Stage 2 (real fine-tuning, **use this for evaluation**)

Run **evaluate.ipynb** next to measure PSNR/SSIM across all stages.
